# 2026-11-25 ARIMA vs seasonal naive on FOODS (raw data)
- Same 30 store × dept series, same 28-day test, same MASE as the baselines notebook.
- Main model: plain ARIMA on the **weekly-differenced** data (today minus the same weekday last week), then back to units.
- Appendix A5: plain ARIMA on the raw data (it differences day to day itself). It did worse, so it moved there.
- One ARIMA per series: 30 local models.

## Step 1: set up

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings
import numpy as np, pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsforecast.models import AutoARIMA
from statsforecast.arima import arima_string
from statsmodels.tsa.stattools import acf, pacf
from m5food import data, plots
plots.set_style()

HORIZON = 28                 # test days
M = 7                        # season length: one week
SCALE_DAYS = 365             # MASE yardstick: last year of training
PLOT_START = '2016-02-01'    # every 30-panel chart starts here, so raw and forecast charts show the same dates

sales = data.add_readable_names(data.load_sales_foods())
cal = data.load_calendar()
days = data.day_columns(sales)
idx = pd.DatetimeIndex(cal.set_index('d').loc[days, 'date'].values, name='date')
series = sales.groupby(['store', 'dept'])[days].sum().T.set_axis(idx)       # 30 columns: (store, dept)
train, test = series.iloc[:-HORIZON], series.iloc[-HORIZON:]

stores = sorted(series.columns.get_level_values(0).unique())
depts = sorted(series.columns.get_level_values(1).unique())
STATE_COLOR = {'California': plots.PALETTE[0], 'Texas': plots.PALETTE[1], 'Wisconsin': plots.PALETTE[2]}
state_of = lambda store: store.split(' store ')[0]
print(f'train: {train.index[0].date()} to {train.index[-1].date()} | test: {test.index[0].date()} to {test.index[-1].date()}')

In [ ]:
def grid(lines, title, subtitle):
    """30 panels (rows = stores, columns = depts), from PLOT_START to the last test day. Grey = test days.
    lines = {legend name: (DataFrame with the 30 columns, line style)}. color='state' = colour by state."""
    fig = make_subplots(rows=len(stores), cols=len(depts), shared_xaxes=True,
                        column_titles=[f'<b>{d}</b>' for d in depts], row_titles=stores,
                        vertical_spacing=0.012, horizontal_spacing=0.04)
    for r, store in enumerate(stores, start=1):
        for c, dept in enumerate(depts, start=1):
            for name, (df, style) in lines.items():
                y = df[(store, dept)].loc[PLOT_START:]
                line = dict(style)
                if line.get('color') == 'state':
                    line['color'] = STATE_COLOR[state_of(store)]
                fig.add_trace(go.Scatter(x=y.index, y=y, name=name, line=line, mode='lines',
                                         showlegend=(r == 1 and c == 1 and len(lines) > 1),
                                         hovertemplate=f'{store}, {dept}<br>%{{x|%a %d %b}}: %{{y:,.0f}}<extra>{name}</extra>'),
                              row=r, col=c)
    fig.add_vrect(x0=test.index[0], x1=test.index[-1], fillcolor='#e6e6e6', opacity=0.6, layer='below',
                  line_width=0, row='all', col='all')
    fig.update_yaxes(nticks=3, tickfont=dict(size=9))
    fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
    fig.update_layout(title=f'<b>{title}</b><br><span style="font-size:13px;color:#444">{subtitle}</span>',
                      template='simple_white', height=1500, margin=dict(t=110, r=150),
                      legend=dict(orientation='h', y=1.03, x=1, xanchor='right'))
    fig.show()

grid({'units a day': (series, dict(color='state', width=1.2))},
     'The 30 raw series: 10 stores × 3 food departments',
     f'Units sold per day, {pd.Timestamp(PLOT_START):%d %b} to {test.index[-1]:%d %b %Y}. Grey = the 28 test days. '
     'Colour = state (blue California, orange Texas, green Wisconsin). Each panel has its own scale.')

## Step 2: seasonal naive 7, the number to beat

In [ ]:
recent = train.iloc[-SCALE_DAYS:]
scale = (recent - recent.shift(M)).abs().mean()          # per series: average miss of "same weekday last week", last year

def mase(forecast, days=slice(None)):
    """MASE per series. forecast in units, same shape as test. days = which test days to score (default all 28)."""
    return (test - forecast).iloc[days].abs().mean() / scale

snaive7 = pd.DataFrame(np.tile(train.iloc[-M:].to_numpy(), (HORIZON // M, 1)), index=test.index, columns=train.columns)
print(f'Seasonal naive 7: global MASE {mase(snaive7).mean():.2f}')

## Step 3: weekly difference
- Change = units today minus units on the same weekday last week.
- Plain ARIMA has no weekly term. The weekly difference holds the week for it.
- ARIMA forecasts the change. We add it back to get units.

In [ ]:
train_wdiff = (train - train.shift(M)).dropna()        # first week drops out: nothing to subtract
print('raw:', train.shape, '| weekly-differenced:', train_wdiff.shape)

## Step 4: ARIMA on the weekly-differenced data

In [ ]:
def fit_all(frame):
    """Plain ARIMA (no seasonal terms) on every column. AutoARIMA picks (p, d, q) per series.
    Returns the 28-day forecasts and the fitted models."""
    fc, fits = pd.DataFrame(index=test.index, columns=frame.columns, dtype=float), {}
    for col in frame.columns:
        fits[col] = AutoARIMA(season_length=1).fit(frame[col].to_numpy(dtype=float))
        fc[col] = fits[col].predict(h=HORIZON)['mean']
    return fc, fits

def undo_weekly_diff(change):
    """Change forecast -> units. Each week = the week before + the forecast change.
    Week 1 builds on the last real training week, week 2 on forecast week 1, and so on."""
    prev, weeks = train[change.columns].iloc[-M:].to_numpy(), []
    for k in range(HORIZON // M):
        prev = prev + change.iloc[M * k: M * (k + 1)].to_numpy()
        weeks.append(prev)
    return pd.DataFrame(np.vstack(weeks), index=test.index, columns=change.columns)

# Check the back-transform: true test changes in -> true test units out
true_change = (series - series.shift(M)).loc[test.index]
print('Back-transform check, max gap in units:', float((undo_weekly_diff(true_change) - test).abs().max().max()))

arima_change, fits_w = fit_all(train_wdiff)
arima = undo_weekly_diff(arima_change)                 # ARIMA on weekly diff, in units
print('Forecast days below zero:', int((arima < 0).sum().sum()))
print(f'ARIMA on weekly diff: global MASE {mase(arima).mean():.2f}')

## Step 5: what (p, d, q) AutoARIMA chose
- **p** = how many past days (AR terms). **d** = how many more differences it took. **q** = how many past errors (MA terms).
- AutoARIMA (Hyndman's auto.arima): picks d with KPSS tests, then tries orders and keeps the lowest AICc.

In [ ]:
def pdq_table(fits, forecast, title):
    """One row per series: chosen order, AICc, MASE. MASE green = beats seasonal naive 7."""
    t = pd.DataFrame([{'store': s, 'dept': d,
                       'p': fits[(s, d)].model_['arma'][0], 'd': fits[(s, d)].model_['arma'][5],
                       'q': fits[(s, d)].model_['arma'][1], 'model': arima_string(fits[(s, d)].model_).strip(),
                       'AICc': fits[(s, d)].model_['aicc'], 'MASE': mase(forecast)[(s, d)]} for s, d in train.columns])
    colour = ['#c8ecd0' if v < mase(snaive7)[k] else '#f4c6c6' for k, v in zip(train.columns, t['MASE'])]
    fig = go.Figure(go.Table(
        header=dict(values=[f'<b>{c}</b>' for c in t.columns], fill_color='#eeeeee', align='left', font=dict(size=12)),
        cells=dict(values=[t[c] if c not in ('AICc', 'MASE') else t[c].map('{:,.2f}'.format) for c in t.columns],
                   align='left', font=dict(size=12), height=24, fill_color=[['white'] * len(t)] * 7 + [colour])))
    fig.update_layout(title=f'<b>{title}</b><br><span style="font-size:12px;color:#555">'
                            'MASE green = ARIMA beats seasonal naive 7 on that series, red = it loses.</span>',
                      height=110 + 25 * len(t), margin=dict(t=80, l=10, r=10, b=10))
    fig.show()
    print('How often:', ' | '.join(f'({p},{d},{q}) x{n}' for (p, d, q), n in
                                   t.groupby(['p', 'd', 'q']).size().sort_values(ascending=False).items()))
    return t

pdq = pdq_table(fits_w, arima, 'The order AutoARIMA chose for each series (weekly-differenced data)')

**What I see**
- d = 0 in all 30. The weekly difference already made them stationary. No extra differencing.
- All zero mean. The change is expected to be 0 on average.
- Smaller orders than on raw: (2,0,2) x6, (5,0,3) x5, and some tiny ones like (1,0,0).
- No need for 3 to 5 AR terms to fake the week. The weekly difference already holds it.

## Step 6: forecasts vs actual, 30 series

In [ ]:
grid({'actual': (series, dict(color='black', width=1.1)),
      'seasonal naive 7': (snaive7, dict(color=plots.PALETTE[0], width=1.5, dash='dot')),
      'ARIMA on weekly diff': (arima, dict(color=plots.PALETTE[3], width=1.5, dash='dash'))},
     'ARIMA on weekly diff and seasonal naive 7 vs actual, 30 series',
     f'Same dates as the raw chart. Grey = the 28 test days. Each panel has its own scale.')

## Step 7: how did ARIMA do?

In [ ]:
score = {'seasonal naive 7': mase(snaive7).mean(), 'ARIMA on weekly diff': mase(arima).mean()}
wins = (mase(arima) < mase(snaive7)).sum()
improve = (1 - score['ARIMA on weekly diff'] / score['seasonal naive 7']) * 100
print(f"ARIMA improves by {improve:.1f}% (MASE {score['ARIMA on weekly diff']:.3f} vs seasonal naive 7 {score['seasonal naive 7']:.3f})")

fig = go.Figure(go.Bar(x=list(score), y=list(score.values()), text=[f'{v:.2f}' for v in score.values()],
                       textposition='outside', marker_color=[plots.PALETTE[0], plots.PALETTE[3]],
                       hovertemplate='%{x}<br>MASE %{y:.2f}<extra></extra>'))
fig.add_hline(y=1, line=dict(color='#333', dash='dot'))
fig.update_layout(title=f'<b>ARIMA on weekly diff improves by {improve:.1f}% over seasonal naive 7</b><br><span style="font-size:13px;color:#444">'
                        f'Average MASE over 30 series, lower is better. ARIMA wins {wins} of 30 series.</span>',
                  yaxis=dict(title='average MASE', range=[0, max(score.values()) * 1.25]),
                  template='simple_white', height=420, width=600, margin=dict(t=100))
fig.show()

**What I see**
- **ARIMA improves by 4.1%**: MASE 1.015 vs seasonal naive 7 at 1.058.
- ARIMA wins **18 of 30**.
- Better than plain ARIMA on raw (1.07, 15 of 30). See A5.
- Small gain only. 0.04 is inside the "too close to call" line. A second test window would settle it.

## Conclusion
- **ARIMA on weekly diff improves by 4.1%** over seasonal naive 7 (global MASE 1.015 vs 1.058), and wins 18 of 30 series.
- Global number = average MASE over the 30 series. MASE has no unit, so big and small series count the same, and below 1 means better than "same weekday last week".
- Scale = last training year, because store jumps make a longer yardstick unfair. Seasonal naive 7 is the number to beat, because it is the best baseline.

# Appendix: more detail, ARIMA on weekly diff vs seasonal naive 7
### A1: MASE per series, side by side

In [ ]:
per_series = pd.DataFrame({'seasonal naive 7': mase(snaive7), 'ARIMA': mase(arima)})
gap = (per_series['ARIMA'] - per_series['seasonal naive 7']).sort_values()
labels = [f'{s} · {d}' for s, d in gap.index]
fig = go.Figure(go.Bar(x=gap, y=labels, orientation='h', marker_color=np.where(gap < 0, '#4c9f70', '#c0504d'),
                       text=[f'{a:.2f} vs {b:.2f}' for a, b in zip(per_series.loc[gap.index, 'ARIMA'],
                                                                  per_series.loc[gap.index, 'seasonal naive 7'])],
                       textposition='outside', textfont=dict(size=9),
                       hovertemplate='%{y}<br>ARIMA minus seasonal naive: %{x:+.2f}<extra></extra>'))
fig.add_vline(x=0, line=dict(color='#333', width=1))
fig.update_yaxes(autorange='reversed', tickfont=dict(size=10))
fig.update_xaxes(title='MASE gap (left = ARIMA better)', range=[gap.min() - 0.35, gap.max() + 0.35])
fig.update_layout(title='<b>ARIMA on weekly diff minus seasonal naive 7, per series</b><br><span style="font-size:13px;color:#444">'
                        'Green = ARIMA wins. Label = ARIMA MASE vs seasonal naive MASE.</span>',
                  template='simple_white', height=900, margin=dict(t=100, l=240))
fig.show()

### A2: error by test week

In [ ]:
weeks = {f'week {k + 1}': slice(7 * k, 7 * (k + 1)) for k in range(HORIZON // M)}
models = {'seasonal naive 7': snaive7, 'ARIMA': arima}
by_week = pd.DataFrame({n: {w: mase(f, sl).mean() for w, sl in weeks.items()} for n, f in models.items()})
by_week_bias = pd.DataFrame({n: {w: (f.iloc[sl].to_numpy().sum() / test.iloc[sl].to_numpy().sum() - 1) * 100
                                 for w, sl in weeks.items()} for n, f in models.items()})
fig = make_subplots(rows=1, cols=2, subplot_titles=['<b>MASE</b> (lower is better)', '<b>Bias %</b> (0 = right level)'])
for name, color, dash in [('seasonal naive 7', plots.PALETTE[0], 'dot'), ('ARIMA', plots.PALETTE[3], 'dash')]:
    fig.add_trace(go.Scatter(x=list(weeks), y=by_week[name], name=name, mode='lines+markers+text',
                             text=by_week[name].map('{:.2f}'.format), textposition='top center',
                             line=dict(color=color, width=2, dash=dash)), row=1, col=1)
    fig.add_trace(go.Scatter(x=list(weeks), y=by_week_bias[name], name=name, showlegend=False, mode='lines+markers+text',
                             text=by_week_bias[name].map('{:+.0f}%'.format), textposition='top center',
                             line=dict(color=color, width=2, dash=dash)), row=1, col=2)
fig.add_hline(y=0, line=dict(color='#888', width=1), row=1, col=2)
fig.update_layout(title='<b>How the error changes over the 4 test weeks</b><br><span style="font-size:13px;color:#444">'
                        'Average over 30 series. Weeks 2 and 3 hold the SNAP days.</span>',
                  template='simple_white', height=430, margin=dict(t=110),
                  legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'))
fig.show()

**What I see**
- Week 1: a tie (0.75 vs 0.76). Both lean on the real last week.
- Weeks 2 and 3 (SNAP): ARIMA a bit better (1.22 vs 1.32, 1.24 vs 1.31). It sits a little higher.
- Week 4: a tie (0.85). The weekly shape holds to the end, unlike ARIMA on raw.
- Both still too low in weeks 2 and 3: ARIMA −8% and −14%, seasonal naive −11% and −16%.

### A3: by dept and by state

In [ ]:
groups = {'by dept': per_series.index.get_level_values('dept'),
          'by state': per_series.index.get_level_values('store').map(state_of)}
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08, subplot_titles=[f'<b>{g}</b>' for g in groups])
for j, key in enumerate(groups.values(), start=1):
    avg = per_series.groupby(key).mean()
    won = (per_series['ARIMA'] < per_series['seasonal naive 7']).groupby(key).sum()
    size = per_series.groupby(key).size()
    for name, color, pattern in [('seasonal naive 7', plots.PALETTE[0], ''), ('ARIMA', plots.PALETTE[3], '/')]:
        fig.add_trace(go.Bar(x=[f'{g}<br>ARIMA wins {won[g]} of {size[g]}' for g in avg.index], y=avg[name], name=name,
                             showlegend=(j == 1), marker_color=color, marker_pattern_shape=pattern,
                             text=avg[name].map('{:.2f}'.format), textposition='outside'), row=1, col=j)
    fig.add_hline(y=1, line=dict(color='#333', dash='dot'), row=1, col=j)
fig.update_yaxes(title='average MASE', range=[0, 1.7])
fig.update_layout(title='<b>Average MASE by dept and by state</b>', barmode='group', bargap=0.25,
                  template='simple_white', height=470, margin=dict(t=90),
                  legend=dict(orientation='h', y=-0.2, x=0.5, xanchor='center'))
fig.show()

**What I see**
- ARIMA is lower or equal in every dept and every state. No group where it clearly loses.
- Biggest gain: Food dept 1 (1.29 vs 1.39) and California (0.85 vs 0.92).
- Texas is a tie (1.20). Wins are spread out: 5 to 8 per group.

### A4: ACF and PACF per series, next to the order AutoARIMA chose
- Computed on the **weekly-differenced** training data (plus any extra d AutoARIMA took), last year only. That is what p and q describe.
- **ACF** = how much today looks like k days ago. **PACF** = the same, but with the days in between taken out.
- Rough reading (Hyndman 9.5): PACF cuts off after lag p → AR(p). ACF cuts off after lag q → MA(q). With both p and q above 0, neither cuts off cleanly.
- Grey band = noise level. Red bars = lags 7, 14, 21, 28 (the week).

In [ ]:
LAGS = 28
def acf_grid(frame, orders, title):
    """ACF and PACF, lags 1-28, one row per series. frame = the data ARIMA was fitted on; orders = pdq table."""
    fig = make_subplots(rows=len(frame.columns), cols=2, horizontal_spacing=0.06, vertical_spacing=0.006,
                        column_titles=['<b>ACF</b>', '<b>PACF</b>'],
                        row_titles=[f'{s} · {d}<br><b>ARIMA({p},{dd},{q})</b>'
                                    for (s, d), p, dd, q in zip(frame.columns, orders['p'], orders['d'], orders['q'])])
    lags = np.arange(1, LAGS + 1)
    week = np.isin(lags, [7, 14, 21, 28])
    for r, col in enumerate(frame.columns, start=1):
        y = frame[col].iloc[-SCALE_DAYS:].to_numpy(dtype=float)
        for _ in range(int(orders.loc[r - 1, 'd'])):
            y = np.diff(y)
        band = 1.96 / np.sqrt(len(y))
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            vals = [acf(y, nlags=LAGS)[1:], pacf(y, nlags=LAGS)[1:]]
        for c, v in enumerate(vals, start=1):
            fig.add_trace(go.Bar(x=lags, y=v, marker_color=np.where(week, '#c0504d', '#5b7a99'), showlegend=False,
                                 hovertemplate=f'{col[0]}, {col[1]}<br>lag %{{x}}: %{{y:.2f}}<extra></extra>'), row=r, col=c)
            fig.add_hrect(y0=-band, y1=band, fillcolor='#bbbbbb', opacity=0.35, line_width=0, row=r, col=c)
    fig.update_yaxes(range=[-1, 1], nticks=3, tickfont=dict(size=8))
    fig.update_xaxes(tickvals=[1, 7, 14, 21, 28], tickfont=dict(size=8))
    fig.update_annotations(selector=dict(textangle=90), textangle=0, xanchor='left', font=dict(size=10))
    fig.update_layout(title=f'<b>{title}</b><br><span style="font-size:13px;color:#444">Lags 1 to 28 days, last training '
                            'year. Red = weekly lags. Grey band = noise level.</span>',
                      template='simple_white', height=140 * len(frame.columns), margin=dict(t=110, r=230))
    fig.show()

acf_grid(train_wdiff, pdq, 'ACF and PACF for each series, weekly-differenced data')

**What I see**
- One clear pattern: a big **negative** spike at lag 7, in all 30 (ACF about −0.41). PACF goes negative at 7, 14 too.
- That's the sign of a weekly difference: it leaves a "pay back last week" effect at lag 7.
- Plain ARIMA can't use lag 7. A seasonal MA term at lag 7 is made for exactly this (SARIMA with (0,1,1) at 7).
- Lags 1 to 6: small spikes, about 3 of 6 above noise. Hence the small p and q.

### A5: plain ARIMA on the raw data (daily difference)
- Same AutoARIMA, fitted on raw units. It chose its own day-to-day difference (d = 1).
- Moved here because it did worse than the weekly-diff version.

In [ ]:
arima_daily, fits_d = fit_all(train)
pdq_daily = pdq_table(fits_d, arima_daily, 'The order AutoARIMA chose for each series (raw data)')

score3 = {'seasonal naive 7': mase(snaive7).mean(), 'ARIMA on raw<br>(daily difference)': mase(arima_daily).mean(),
          'ARIMA on weekly diff': mase(arima).mean()}
wins_daily = (mase(arima_daily) < mase(snaive7)).sum()
fig = go.Figure(go.Bar(x=list(score3), y=list(score3.values()), text=[f'{v:.2f}' for v in score3.values()],
                       textposition='outside', marker_color=[plots.PALETTE[0], plots.PALETTE[1], plots.PALETTE[3]],
                       hovertemplate='%{x}<br>MASE %{y:.2f}<extra></extra>'))
fig.add_hline(y=1, line=dict(color='#333', dash='dot'))
fig.update_layout(title='<b>Daily vs weekly difference for ARIMA</b><br><span style="font-size:13px;color:#444">'
                        f'Average MASE over 30 series. Wins vs seasonal naive 7: daily {wins_daily} of 30, '
                        f'weekly {wins} of 30.</span>',
                  yaxis=dict(title='average MASE', range=[0, max(score3.values()) * 1.25]),
                  template='simple_white', height=420, width=700, margin=dict(t=100))
fig.show()

In [ ]:
grid({'actual': (series, dict(color='black', width=1.1)),
      'ARIMA on raw (daily difference)': (arima_daily, dict(color=plots.PALETTE[1], width=1.5, dash='dash')),
      'ARIMA on weekly diff': (arima, dict(color=plots.PALETTE[3], width=1.5, dash='dashdot'))},
     'Daily vs weekly difference for ARIMA, 30 series',
     'Same dates as the raw chart. Grey = the 28 test days. Each panel has its own scale.')

In [ ]:
acf_grid(train, pdq_daily, "ACF and PACF for each series, raw data after AutoARIMA's daily difference")

**What I see**
- On raw data AutoARIMA chose d = 1 everywhere, and p of 2 to 5.
- Global 1.07, wins 15 of 30. Worse than the weekly-diff version (1.02, 18 of 30).
- Its ACF after the daily difference still has the week in it (~+0.35 at 7, 14, 21, 28). Plain ARIMA can't hold that, so the forecast fades flat.

### A6: p and q read from the ACF / PACF by eye, vs AutoARIMA
Same data as A4: weekly-differenced once (lag 7), so d = 0. Only lags 1 to 6 decide p and q.

**The rule I use by eye** (Hyndman 9.5), written down so it is the same for all 30:
1. Count the PACF bars outside the grey band, from lag 1 in a row, until the first one inside. Call it **P**.
2. Same for the ACF. Call it **A**.
3. PACF stops first (P < A), ACF fades → **AR: p = P, q = 0**.
4. ACF stops first (A < P), PACF fades → **MA: p = 0, q = A**.
5. Both stop at the same lag → can't tell → **p = 1, q = 1**.
6. Cap at 5. Ignore lag 7 and up (plain ARIMA can't use them).

In [ ]:
from statsforecast.models import ARIMA

def run_length(values, band):
    """How many lags in a row, from lag 1, sit outside the noise band."""
    n = 0
    for v in values:
        if abs(v) <= band:
            break
        n += 1
    return n

eye = []
for col in train.columns:
    y = train_wdiff[col].iloc[-SCALE_DAYS:].to_numpy(dtype=float)       # same data as the A4 chart
    band = 1.96 / np.sqrt(len(y))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        P = min(run_length(pacf(y, nlags=6)[1:], band), 5)
        A = min(run_length(acf(y, nlags=6)[1:], band), 5)
    p, q = (P, 0) if P < A else (0, A) if A < P else (min(P, 1), min(A, 1))
    eye.append({'store': col[0], 'dept': col[1], 'PACF run': P, 'ACF run': A, 'p': p, 'q': q})
eye = pd.DataFrame(eye)

# Fit plain ARIMA(p, 0, q) with those orders on the weekly-differenced data, zero mean (same as AutoARIMA chose)
eye_change = pd.DataFrame(index=test.index, columns=train.columns, dtype=float)
eye_aicc = {}
for i, col in enumerate(train.columns):
    m = ARIMA(order=(int(eye.p[i]), 0, int(eye.q[i])), include_mean=False).fit(train_wdiff[col].to_numpy(dtype=float))
    eye_change[col] = m.predict(h=HORIZON)['mean']
    eye_aicc[col] = m.model_['aicc']
arima_eye = undo_weekly_diff(eye_change)

eye['by eye'] = [f'({p},0,{q})' for p, q in zip(eye.p, eye.q)]
eye['AutoARIMA'] = [f'({p},0,{q})' for p, q in zip(pdq.p, pdq.q)]
eye['MASE by eye'] = mase(arima_eye).to_numpy()
eye['MASE auto'] = mase(arima).to_numpy()
eye['AICc by eye'] = list(eye_aicc.values())
eye['AICc auto'] = pdq['AICc'].to_numpy()
print('How often (by eye):', eye['by eye'].value_counts().to_dict())

In [ ]:
show = eye[['store', 'dept', 'PACF run', 'ACF run', 'by eye', 'AutoARIMA', 'MASE by eye', 'MASE auto']]
better = np.where(show['MASE by eye'] < show['MASE auto'], '#c8ecd0', '#f4c6c6')
fig = go.Figure(go.Table(
    header=dict(values=[f'<b>{c}</b>' for c in show.columns], fill_color='#eeeeee', align='left', font=dict(size=12)),
    cells=dict(values=[show[c].map('{:.2f}'.format) if 'MASE' in c else show[c] for c in show.columns],
               align='left', font=dict(size=12), height=24,
               fill_color=[['white'] * len(show)] * 6 + [better, ['white'] * len(show)])))
fig.update_layout(title='<b>p and q by eye vs AutoARIMA, weekly-differenced data</b><br><span style="font-size:12px;color:#555">'
                        'MASE by eye green = beats AutoARIMA on that series, red = loses.</span>',
                  height=110 + 25 * len(show), margin=dict(t=80, l=10, r=10, b=10))
fig.show()

score4 = {'seasonal naive 7': mase(snaive7).mean(), 'ARIMA, p q by eye': mase(arima_eye).mean(),
          'AutoARIMA': mase(arima).mean()}
fig = go.Figure(go.Bar(x=list(score4), y=list(score4.values()), text=[f'{v:.2f}' for v in score4.values()],
                       textposition='outside', marker_color=[plots.PALETTE[0], plots.PALETTE[2], plots.PALETTE[3]],
                       hovertemplate='%{x}<br>MASE %{y:.2f}<extra></extra>'))
fig.add_hline(y=1, line=dict(color='#333', dash='dot'))
fig.update_layout(title='<b>By eye vs AutoARIMA (both on weekly diff)</b><br><span style="font-size:13px;color:#444">'
                        f'Average MASE over 30 series. By eye beats AutoARIMA in '
                        f'{(eye["MASE by eye"] < eye["MASE auto"]).sum()} of 30, beats seasonal naive 7 in '
                        f'{(mase(arima_eye) < mase(snaive7)).sum()} of 30.</span>',
                  yaxis=dict(title='average MASE', range=[0, max(score4.values()) * 1.25]),
                  template='simple_white', height=420, width=700, margin=dict(t=100))
fig.show()
print(f"AICc lower for AutoARIMA in {(eye['AICc auto'] < eye['AICc by eye']).sum()} of 30 "
      "(AICc = in-sample fit; MASE = the 28 test days)")

**What I see**
- By eye gives small, simple orders: (1,0,0) x13, (1,0,1) x9, (2,0,0) x5, (4,0,0) x3. Mostly "PACF stops at lag 1, ACF fades" = AR(1).
- AutoARIMA picks bigger ones: (2,0,2), (5,0,3) and so on.
- By eye: global MASE **1.07**. AutoARIMA: **1.02**. Seasonal naive 7: 1.06.
- By eye beats AutoARIMA in only 12 of 30, and seasonal naive 7 in 12 of 30.
- Biggest gaps: the Wisconsin store 2 series. There ACF and PACF both run long (4 to 5 lags), a mixed AR + MA case. A pure AR(4) misses it.
- AutoARIMA also has the lower AICc in 29 of 30. Its search fits the training data better, and here that carried over to the test.
- So the chart is good for a first guess and for learning. The search (try nearby orders, keep lowest AICc) does the last step better.

### A7: AR, MA and ARMA vs AutoARIMA vs seasonal naive 7
- Same weekly-differenced data (d = 0), same back-transform, same MASE.
- The three textbook shapes, one fixed order for all 30 series:
  - **AR(1)** = (1,0,0): today's change follows yesterday's change.
  - **MA(1)** = (0,0,1): today's change follows yesterday's surprise.
  - **ARMA(1,1)** = (1,0,1): both.
- **AutoARIMA** = its own (p, q) per series, picked by AICc.

In [ ]:
fixed = {'AR(1)': (1, 0, 0), 'MA(1)': (0, 0, 1), 'ARMA(1,1)': (1, 0, 1)}
family = {}
for name, order in fixed.items():
    change = pd.DataFrame(index=test.index, columns=train.columns, dtype=float)
    for col in train.columns:
        change[col] = (ARIMA(order=order, include_mean=False)
                       .fit(train_wdiff[col].to_numpy(dtype=float)).predict(h=HORIZON)['mean'])
    family[name] = undo_weekly_diff(change)

all_models = {'seasonal naive 7': snaive7, **family, 'AutoARIMA': arima}
family_mase = pd.DataFrame({n: mase(f) for n, f in all_models.items()})
summary = pd.DataFrame({'global MASE': family_mase.mean(),
                        'wins vs seasonal naive 7': (family_mase.lt(family_mase['seasonal naive 7'], axis=0)).sum(),
                        'best of the 5 in': family_mase.idxmin(axis=1).value_counts().reindex(family_mase.columns, fill_value=0)})
summary.loc['seasonal naive 7', 'wins vs seasonal naive 7'] = np.nan

colors = [plots.PALETTE[0], '#8fb3cf', '#c9a3b8', plots.PALETTE[2], plots.PALETTE[3]]
fig = go.Figure(go.Bar(x=summary.index, y=summary['global MASE'], marker_color=colors,
                       text=[f'{v:.2f}' for v in summary['global MASE']], textposition='outside',
                       hovertemplate='%{x}<br>MASE %{y:.2f}<extra></extra>'))
fig.add_hline(y=1, line=dict(color='#333', dash='dot'))
fig.update_layout(title='<b>AR, MA, ARMA and AutoARIMA vs seasonal naive 7</b><br><span style="font-size:13px;color:#444">'
                        'All ARIMA models on weekly-differenced data, scored in units. Average MASE over 30, lower is better.</span>',
                  yaxis=dict(title='average MASE', range=[0, summary['global MASE'].max() * 1.25]),
                  template='simple_white', height=430, width=760, margin=dict(t=100))
fig.show()
summary.round(2)

In [ ]:
# Per series: which of the 5 is best (darkest green = lowest MASE in that row)
rank = family_mase.rank(axis=1)
labels = [f'{s} · {d}' for s, d in family_mase.index]
fig = go.Figure(go.Heatmap(z=rank.to_numpy(), x=family_mase.columns, y=labels, zmin=1, zmax=5,
                           colorscale=[[0, '#4c9f70'], [0.5, '#f7f7f7'], [1, '#e8a3a3']], showscale=False,
                           text=family_mase.round(2).to_numpy(), texttemplate='%{text}', textfont=dict(size=10),
                           xgap=2, ygap=2, hovertemplate='%{y}<br>%{x}: MASE %{text}<extra></extra>'))
fig.update_yaxes(autorange='reversed')
fig.update_layout(title='<b>MASE per series, 5 models</b><br><span style="font-size:13px;color:#444">'
                        'Colour = rank in the row: green = best of the 5, red = worst. Number = MASE.</span>',
                  template='simple_white', height=950, margin=dict(t=100, l=260))
fig.show()

**What I see**
- AR(1) 1.06, MA(1) 1.06. The same as seasonal naive 7 (1.06).
- Why: their forecast of the change fades to 0 within a few days. A change of 0 = seasonal naive 7. So after week 1 they just repeat week 1: seasonal naive plus a small fixed shift.
- ARMA(1,1) is worse: 1.09. One extra term, no extra skill. It goes badly wrong on a few series (Wisconsin store 2, Food dept 2: 1.30).
- AutoARIMA is the only one that gains: **1.02**, wins 18 of 30, best of the 5 in 12 series.
- So the gain comes from the bigger orders AutoARIMA picks per series, not from "using ARIMA" as such.
- Small models only copy the last week. To beat seasonal naive, the model has to learn more than one step back.

### A8: ARIMA on 5, 3 and 2 years of training
- Same model as the main part: AutoARIMA on the weekly-differenced data, back to units.
- Only the training window changes: the last 5, 3 or 2 years before the test.
- The MASE scale stays the same (last training year), so the numbers compare directly.

In [ ]:
windows = {'5 years': 5 * 365, '3 years': 3 * 365, '2 years': 2 * 365}
by_window = {}
for name, n in windows.items():
    fc, _ = fit_all(train_wdiff.iloc[-n:])
    by_window[name] = undo_weekly_diff(fc)

win_mase = pd.DataFrame({n: mase(f) for n, f in {'seasonal naive 7': snaive7, **by_window}.items()})
win_table = pd.DataFrame({'global MASE': win_mase.mean(),
                          'vs seasonal naive 7': (1 - win_mase.mean() / win_mase['seasonal naive 7'].mean()) * 100,
                          'wins vs seasonal naive 7': win_mase.lt(win_mase['seasonal naive 7'], axis=0).sum()})
win_table.loc['seasonal naive 7', ['vs seasonal naive 7', 'wins vs seasonal naive 7']] = np.nan
print('Full training (5.2 years, main result):', f'{mase(arima).mean():.3f}')
win_table.round(3)

**What I see**
- More history helps here. 5 years: improves by 4.7%. 3 years: 3.3%. 2 years: 2.5%.
- Full training (5.2 years, main result): 4.1%. About the same as 5 years.
- So the store jumps don't spoil the long window. The weekly difference already takes the old level out.
- Keep the full window as the main result. Small differences, one test window.